# Attention, Positional Encoding, Decoder Block
- https://newsletter.theaiedge.io/p/the-transformer-architecture-v2

### 1. Attention Mechanism


**Objective**: Enable models to focus on relevant parts of input sequences, essential for handling longer dependencies.

**Explanation**: 
- Attention allows a model to “attend” to specific input parts when generating each output. It’s especially useful for tasks requiring selective referencing of input tokens (e.g., translation, summarization).

    

In [16]:
import torch
import torch.nn as nn

# Simple attention mechanism
def attention(query, key, value):
    scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(key.size(-1), dtype=torch.float32))
    attn_weights = torch.nn.functional.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, value)
    return output, attn_weights

# Example inputs
query = torch.randn(5, 3, 8)  # Batch size = 5, Sequence length = 3, Embedding size = 8
key = torch.randn(5, 3, 8)
value = torch.randn(5, 3, 8)

# Apply attention
output, weights = attention(query, key, value)
print("Attention Output:", output)
print("Attention Weights:", weights)


Attention Output: tensor([[[ 0.3015, -0.5424,  0.0835,  1.2219,  0.1377,  0.4951, -0.4969,
           0.0360],
         [ 0.2827, -0.5684, -0.7148,  0.8956,  0.5841,  0.8953, -0.3201,
           0.0189],
         [-0.0031, -0.2527, -0.5818,  1.0131,  0.3155,  0.5806, -0.2713,
           0.0852]],

        [[-1.0567,  0.5040, -0.5084,  1.3888,  0.0591,  0.0168,  0.0583,
          -0.8553],
         [-1.0431,  0.4480, -0.4843,  1.2783,  0.2435,  0.1115,  0.1365,
          -0.7885],
         [-0.3406,  0.9173, -0.2775,  1.1220,  0.3476,  0.5178, -0.7775,
          -1.0706]],

        [[-0.4773,  0.2585,  0.5964,  0.0404, -0.4971, -0.0343, -0.6698,
          -0.2367],
         [-1.5521, -0.5000,  1.1853, -0.1870,  1.4317, -0.5948, -1.4801,
           1.4884],
         [-0.4518,  0.4094,  0.4257, -0.0148, -0.6157,  0.1141, -0.9601,
          -0.0489]],

        [[-0.5149, -0.2957, -0.3092, -0.3433,  0.7758, -0.1406, -0.1414,
           0.6573],
         [-0.4153,  0.0397, -0.2985, -0.4699, 

### 2. Self-Attention Mechanism


**Objective**: Understand how self-attention allows a model to relate each word in a sentence to every other word, helping capture context effectively.

**Explanation**:
   - **Queries, Keys, and Values**: In self-attention, each word is represented by three vectors: a query, a key, and a value.
   - **Dot Product and Scaling**: The query and key vectors are multiplied to determine attention scores, representing the similarity between words. These scores are scaled and then passed through a softmax function to obtain attention weights.
   - **Weighted Sum**: Each word’s final representation is the sum of all value vectors, weighted by attention scores, allowing the model to focus on relevant words in context.
    

### 3. Multi-Head Attention


**Objective**: Enhance the model’s ability to capture diverse word relationships by using multiple attention heads.

**Explanation**:
   - **Multiple Heads**: Multi-head attention applies self-attention multiple times in parallel, each with different learned projections. This allows the model to capture various types of dependencies, such as grammatical or semantic.
   - **Concatenation and Linear Transformation**: Each head’s output is concatenated and passed through a linear layer, blending the insights from all heads into a single representation.
    

In [18]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert embed_size % num_heads == 0
        self.head_dim = embed_size // num_heads
        self.num_heads = num_heads

        # Linear layers for queries, keys, values
        self.query = nn.Linear(embed_size, embed_size)
        self.key = nn.Linear(embed_size, embed_size)
        self.value = nn.Linear(embed_size, embed_size)
        
        # Final linear layer after concatenation
        self.fc_out = nn.Linear(embed_size, embed_size)

    def forward(self, query, key, value):
        batch_size, seq_len, embed_size = query.size()

        # Transform inputs into multiple heads
        query = self.query(query).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        key = self.key(key).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        value = self.value(value).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Apply self-attention
        scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.head_dim, dtype=torch.float32))
        attn_weights = F.softmax(scores, dim=-1)
        attention = torch.matmul(attn_weights, value)
        
        # Concatenate heads and pass through the final linear layer
        attention = attention.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_size)
        return self.fc_out(attention)

# Example usage
embed_size = 16
num_heads = 4
x = torch.randn(2, 5, embed_size)  # Batch size = 2, Sequence length = 5

multi_head_attention = MultiHeadAttention(embed_size, num_heads)
output = multi_head_attention(x, x, x)
print("Multi-Head Attention Output:", output)


Multi-Head Attention Output: tensor([[[ 0.2662, -0.2156, -0.2180, -0.1860,  0.0863, -0.2513,  0.0595,
          -0.1095,  0.1051,  0.0735,  0.1050,  0.3097,  0.0336, -0.0251,
          -0.1898,  0.1396],
         [ 0.2814, -0.1807, -0.1311, -0.1531,  0.0244, -0.3471,  0.1419,
          -0.0202,  0.1131,  0.1463,  0.1270,  0.3574,  0.1143,  0.0729,
          -0.2908,  0.0815],
         [ 0.2375, -0.1649, -0.2115, -0.1225,  0.0463, -0.3216,  0.1305,
          -0.0386,  0.1059,  0.1388,  0.1416,  0.3304,  0.0616,  0.0288,
          -0.2749,  0.1335],
         [ 0.2144, -0.2204, -0.2061, -0.1153,  0.0691, -0.3267,  0.1245,
          -0.0850,  0.1249,  0.0894,  0.1426,  0.3297,  0.0543, -0.0047,
          -0.2668,  0.1798],
         [ 0.2609, -0.1851, -0.1172, -0.0979,  0.0429, -0.3422,  0.0570,
          -0.0695,  0.1175,  0.1385,  0.1207,  0.3771,  0.0941,  0.0470,
          -0.2778,  0.1203]],

        [[ 0.0631,  0.0571,  0.0143,  0.2702,  0.1656, -0.0956,  0.0911,
          -0.0999,  0

### 4. Positional Encoding


**Objective**: Introduce positional information into word embeddings, as Transformers process words in parallel without inherent sequence order.

**Explanation**:
   - Since Transformers do not process words sequentially, positional encodings are added to the embeddings to give the model information about the order of words.
   - **Sine and Cosine Functions**: Positional encodings are computed using sine and cosine functions with varying frequencies, creating unique patterns for each position in the sequence.
    

In [19]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, embed_size, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_size)
        for pos in range(max_len):
            for i in range(0, embed_size, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2 * i) / embed_size)))
                pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * i) / embed_size)))
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

# Example usage
embed_size = 16
seq_len = 10
pos_encoding = PositionalEncoding(embed_size)
x = torch.randn(1, seq_len, embed_size)
output = pos_encoding(x)
print("Positional Encoding Output:", output)


Positional Encoding Output: tensor([[[-1.0514, -0.5359, -0.6516, -1.1832, -1.8144,  0.8563, -0.3979,
           0.9169,  0.7565, -1.1920,  1.2575,  1.7594,  2.0920, -1.5550,
          -0.8826,  0.5562],
         [-0.7210,  2.0523, -0.2256,  0.9716, -0.0100,  0.5223,  0.2362,
          -1.4465, -0.4246,  0.1115, -0.0466, -1.5084,  0.7966,  1.9744,
          -0.4737,  2.0870],
         [ 0.6037, -1.5329,  0.0650, -0.2832, -1.3006,  2.8583,  0.2978,
          -0.6167, -0.4815,  2.2409,  1.4391,  2.2654, -3.1288,  0.8184,
          -0.7322,  1.7395],
         [ 0.3747, -0.8800,  0.5237,  3.4395, -0.0380,  0.6857, -0.6169,
          -0.0092,  0.2644, -0.1020,  1.2138,  2.0708,  2.2180,  1.2193,
           0.3764,  0.7779],
         [-0.7974, -1.5611, -0.9831,  0.8691, -0.0136,  1.4039,  0.3077,
           2.4569,  0.8197,  2.8361,  1.7505,  3.4311, -0.4544,  1.8216,
          -0.5136,  1.2580],
         [-1.2787, -0.3886, -1.4239,  0.8563,  0.2021, -0.7575, -0.1807,
          -0.3424,  0.54

### 5. Transformer Decoder Block


**Objective**: Understand how the Transformer decoder combines self-attention, encoder-decoder attention, and feedforward layers, the fundamental units in a Transformer.

**Explanation**:
   - **Self-Attention Layer**: The decoder starts with a masked multi-head attention layer, allowing the model to attend to relevant parts of the input while preventing attending to future tokens.
   - **Encoder-Decoder Attention**: The second attention layer attends to the encoder’s output, allowing the decoder to focus on relevant parts of the input sequence.
   - **Add & Norm**: After each attention layer, a residual connection is added, followed by layer normalization to stabilize learning.
   - **Feedforward Layer**: The attention output is passed through a fully connected layer for further processing.
   - **Final Add & Norm**: A second residual connection and layer normalization complete the block, making the Transformer’s decoder robust to varying input sequences.
    

In [20]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, embed_size, num_heads, forward_expansion):
        super(TransformerDecoderBlock, self).__init__()
        self.self_attention = MultiHeadAttention(embed_size, num_heads)
        self.norm1 = nn.LayerNorm(embed_size)
        self.cross_attention = MultiHeadAttention(embed_size, num_heads)  # Cross-attention layer
        self.norm2 = nn.LayerNorm(embed_size)
        self.encoder_decoder_attention = MultiHeadAttention(embed_size, num_heads)
        self.norm3 = nn.LayerNorm(embed_size)
        self.norm4 = nn.LayerNorm(embed_size)

        self.feed_forward = nn.Sequential(
            nn.Linear(embed_size, forward_expansion * embed_size),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_size, embed_size)
        )

    def forward(self, x, enc_out):
        self_attention, _ = self.self_attention(x, x, x)
        x = self.norm1(self_attention + x)  # Add & Norm
        cross_attention, _ = self.cross_attention(x, enc_out, enc_out)  # Cross-attention
        x = self.norm2(cross_attention + x)  # Add & Norm
        encoder_decoder_attention, _ = self.encoder_decoder_attention(x, enc_out, enc_out)
        x = self.norm3(encoder_decoder_attention + x)  # Add & Norm
        forward = self.feed_forward(x)
        x = self.norm4(forward + x)  # Add & Norm
        return x

# Example usage
embed_size = 16
num_heads = 4
forward_expansion = 4
decoder_block = TransformerDecoderBlock(embed_size, num_heads, forward_expansion)
x = torch.randn(2, 5, embed_size)
enc_out = torch.randn(2, 5, embed_size)
output = decoder_block(x, enc_out)
print("Transformer Decoder Block Output:", output)


Transformer Decoder Block Output: tensor([[[ 0.4910, -0.6464,  0.8437,  0.7912,  0.8979, -1.4663,  0.2569,
          -0.1934,  0.8170,  1.3415, -1.3216, -0.5290, -2.0322,  0.0062,
          -0.6390,  1.3824],
         [ 0.7071, -0.2541,  1.4562,  0.1670,  0.6184, -1.0367, -1.4088,
           1.6211, -0.5591,  0.6664, -1.0686,  1.0898, -0.7737,  0.8379,
          -0.4388, -1.6242],
         [ 0.4265, -0.6886,  1.4683, -1.4121,  1.7131,  0.2169,  0.3689,
          -1.1298, -1.4095,  0.4197, -1.5089,  0.6499, -0.2969, -0.2936,
           0.1577,  1.3184],
         [ 1.0832,  0.7732,  0.6037, -0.3101,  0.9140, -0.0921, -0.9836,
           1.4108, -0.1122,  0.9556, -0.6135, -0.0088, -1.8232,  1.0673,
          -1.2101, -1.6543],
         [ 0.7539,  1.6730,  0.0353, -1.8893,  0.4281,  0.6037, -0.4903,
          -0.2317,  0.1276, -0.2157,  0.0350,  1.5450, -1.6681,  1.0698,
          -0.5246, -1.2516]],

        [[-1.5950,  0.5712,  0.8801, -0.7915,  1.5770, -0.1645, -0.6999,
          -0.791